# EDA — `bronze.trust_prices_csv`

The archive. Yahoo deletes delisted companies, so this file is the only record the
project holds of the trusts that died.

Two questions only:

1. Are `BCPT` and `CSH` — the two trusts Yahoo refuses — actually usable?
2. Is this file really the dirty source it was described as, or did it inherit its
   defects from Yahoo?

## 1. Shape, and how much of it is actually needed

In [0]:
%sql
SELECT COUNT(*)               AS rows,
       COUNT(DISTINCT ticker) AS tickers,
       MIN(`date`)            AS first_date,
       MAX(`date`)            AS last_date
FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv;

In [0]:
%sql
-- Which tickers here are NOT available from Yahoo. Only these are worth keeping.
SELECT c.ticker,
       COUNT(*)    AS rows,
       MIN(c.`date`) AS first_date,
       MAX(c.`date`) AS last_date
FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv c
LEFT JOIN (
  SELECT DISTINCT source_ticker FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
) y ON y.source_ticker = c.ticker
WHERE y.source_ticker IS NULL
GROUP BY c.ticker
ORDER BY rows DESC;

**16,357 rows across 102 tickers, of which only two are unavailable from Yahoo:**

- `BCPT` — 158 rows, 2011-09 to 2024-11
- `CSH` — 112 rows, 2016-12 to 2026-03

So 270 rows out of 16,357 are load-bearing. Everything else is superseded by a Yahoo
series that is deeper and carries dividends.

This is also the honest scale of the survivorship cohort: **two trusts**. Worth stating
plainly rather than letting "we account for delisted trusts" imply more than it is.

## 2. Are the two trusts that matter clean enough to use?

They have no dividends, so they can only ever contribute a price return. What they must
not have is the scale defect.

In [0]:
%sql
WITH base AS (
  SELECT ticker,
         SUBSTRING(`date`, 1, 7)             AS month_key,
         CAST(price_gbx_or_gbp AS DOUBLE)    AS price
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv
  WHERE ticker IN ('BCPT', 'CSH') AND CAST(price_gbx_or_gbp AS DOUBLE) > 0
),
with_median AS (
  SELECT *,
         PERCENTILE_APPROX(price, 0.5) OVER (
           PARTITION BY ticker ORDER BY month_key
           ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING
         ) AS local_median
  FROM base
)
SELECT ticker,
       COUNT(*)                                                     AS months,
       SUM(CASE WHEN price / local_median > 2
                  OR price / local_median < 0.5 THEN 1 ELSE 0 END)  AS scale_outliers,
       ROUND(MIN(price), 2)                                         AS min_price,
       ROUND(MAX(price), 2)                                         AS max_price
FROM with_median
WHERE local_median > 0
GROUP BY ticker
ORDER BY ticker;

If `scale_outliers` is 0 for both, the two archived trusts can be used as they stand and
need none of the repair the Yahoo table needs.

## 3. Who actually introduced the defects?

This file was described in the original design as the dirty source — mis-scaled rows and
a zero price. Check whether those same defects exist in Yahoo for the same tickers.

In [0]:
%sql
-- CGT in this file, around the months Yahoo also reports ten times too high.
SELECT SUBSTRING(`date`, 1, 7)                    AS month,
       ROUND(CAST(price_gbx_or_gbp AS DOUBLE), 2) AS csv_price
FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv
WHERE ticker = 'CGT' AND `date` >= '2025-01-01' AND `date` < '2025-09-01'
ORDER BY `date`;

In [0]:
%sql
-- The zero price, in both sources, on the same date.
SELECT 'csv' AS source, ticker, `date`, price_gbx_or_gbp AS price
FROM `index-vs-trust-pipeline`.bronze.trust_prices_csv
WHERE ticker = 'PCFT' AND `date` = '2019-11-01'
UNION ALL
SELECT 'yahoo', source_ticker, `Date`, `Close`
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
WHERE source_ticker = 'PCFT' AND `Date` = '2019-11-01 00:00:00';

**The CSV inherited its defects; it did not create them.** `CGT` carries the same 10x
rows here as in Yahoo, and `PCFT` carries the same zero on the same date in both sources.

That retires the "the CSV is the dirty source" story, and it explains why switching to
Yahoo did not make the cleaning go away. The one defect that **is** unique to this file
is the date labelling — month-end mixed with next-month-first — and it is moot, because
only two tickers are used and Yahoo's bars are all month-start anyway.

---

## Findings

| # | Finding | Status |
|---|---|---|
| 2.1 | 16,357 rows, 102 tickers, but only **`BCPT` and `CSH`** are unavailable from Yahoo — 270 rows are load-bearing. | **settled** |
| 2.2 | The delisted cohort really is two trusts. The survivorship claim must be framed as a mechanism, not an effect size. | **settled** |
| 2.3 | Scale-outlier check on `BCPT` and `CSH` — measured above; if 0, they are usable unrepaired. | **settled** |
| 2.4 | The mis-scaled rows and the `PCFT` zero exist in **both** sources, so the CSV inherited them from Yahoo. | **settled** |
| 2.5 | Mixed date labelling is unique to this file but moot, since only two tickers are used. | **settled** |
| 2.6 | These two trusts have no dividends, so they can contribute price return only while everything else is total return. | **open** |

Finding 2.6 is the awkward one for Silver: the two archived trusts cannot be measured on
the same basis as the rest.